# 🎙️ Real-Time Emotion, Age & Gender Recognition from Speech
### Deep Learning with LSTM · RAVDESS Dataset · TensorFlow/Keras

**Project**: VoiceIQ Emotion Recognition System  
**University**: Abdul Wali Khan University Mardan  
**Tech Stack**: Python · Librosa · TensorFlow · Scikit-learn

---

## Notebook Structure
1. Setup & Imports
2. Dataset Overview (RAVDESS)
3. Feature Extraction (MFCC, Chroma, Mel Spectrogram)
4. Data Augmentation
5. Dataset Preparation
6. LSTM Model Architecture
7. Training & Validation
8. Evaluation & Confusion Matrix
9. Save Model Artifacts

## 1. Setup & Imports

In [ ]:
# Install required packages
# !pip install librosa soundfile tensorflow scikit-learn pandas numpy matplotlib seaborn

import os
import glob
import pickle
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f'TensorFlow version: {tf.__version__}')
print(f'Librosa version: {librosa.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 2. Dataset Overview (RAVDESS)

In [ ]:
"""
RAVDESS Dataset Structure:
Filename format: 03-01-06-01-02-01-12.wav
Positions:
  [0] Modality:    01=AV, 02=Video, 03=Audio
  [1] Channel:     01=Speech, 02=Song
  [2] Emotion:     01=Neutral, 02=Calm, 03=Happy, 04=Sad,
                   05=Angry, 06=Fearful, 07=Disgust, 08=Surprised
  [3] Intensity:   01=Normal, 02=Strong
  [4] Statement:   01='Kids are talking', 02='Dogs are sitting'
  [5] Repetition:  01, 02
  [6] Actor:       01-24 (01-12=Male, 13-24=Female)
"""

DATASET_PATH = './ravdess-dataset'  # Update to your RAVDESS path

EMOTION_MAP = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

print('Emotion labels:', list(EMOTION_MAP.values()))

# Count files
audio_files = glob.glob(os.path.join(DATASET_PATH, '**/*.wav'), recursive=True)
print(f'Total audio files found: {len(audio_files)}')

## 3. Feature Extraction

In [ ]:
# Feature extraction configuration
SAMPLE_RATE = 22050
DURATION    = 3.0      # seconds
N_MFCC      = 40
N_CHROMA    = 12
HOP_LENGTH  = 512


def extract_features(file_path: str) -> np.ndarray:
    """
    Extract a comprehensive feature vector from an audio file.
    
    Feature vector composition (162 total features):
    - MFCC mean (40)
    - MFCC std  (40)
    - Chroma    (12)
    - Mel Spec  (60)
    - ZCR       (1)
    - Spectral Contrast (7)
    - Spectral Rolloff  (1)
    - RMS Energy        (1)
    """
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        y = librosa.util.normalize(y)

        # MFCC
        mfcc    = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std  = np.std(mfcc, axis=1)

        # Chroma
        stft    = np.abs(librosa.stft(y))
        chroma  = librosa.feature.chroma_stft(S=stft, sr=sr, n_chroma=N_CHROMA)
        chroma_mean = np.mean(chroma, axis=1)

        # Mel Spectrogram (first 60 bands)
        mel     = librosa.feature.melspectrogram(y=y, sr=sr)
        mel_mean = np.mean(mel, axis=1)[:60]

        # Zero Crossing Rate
        zcr     = np.mean(librosa.feature.zero_crossing_rate(y))

        # Spectral Contrast
        contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sr), axis=1)

        # Spectral Rolloff
        rolloff  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

        # RMS Energy
        rms      = np.mean(librosa.feature.rms(y=y))

        return np.concatenate([
            mfcc_mean, mfcc_std, chroma_mean, mel_mean,
            [zcr], contrast, [rolloff], [rms]
        ]).astype(np.float32)

    except Exception as e:
        print(f'Error processing {file_path}: {e}')
        return None


# Test on one file
if audio_files:
    sample_features = extract_features(audio_files[0])
    print(f'Feature vector shape: {sample_features.shape}')
    print(f'Feature range: [{sample_features.min():.2f}, {sample_features.max():.2f}]')

## 4. Data Augmentation

In [ ]:
def augment_audio(y: np.ndarray, sr: int, technique: str) -> np.ndarray:
    """
    Apply audio augmentation to increase dataset diversity.
    
    Techniques:
    - noise: Add Gaussian white noise
    - shift: Time-shift the waveform
    - pitch: Pitch shift up/down
    - stretch: Time stretch without pitch change
    """
    if technique == 'noise':
        noise = np.random.normal(0, 0.005, y.shape)
        return y + noise

    elif technique == 'shift':
        shift = int(np.random.uniform(-0.2, 0.2) * sr)
        return np.roll(y, shift)

    elif technique == 'pitch':
        n_steps = np.random.choice([-2, -1, 1, 2])
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

    elif technique == 'stretch':
        rate = np.random.uniform(0.8, 1.2)
        return librosa.effects.time_stretch(y, rate=rate)

    return y


def extract_with_augmentation(file_path: str):
    """
    Extract features from original + 2 augmented versions.
    Returns list of (features, label) tuples.
    """
    results = []
    
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        y = librosa.util.normalize(y)
        
        # Original
        feat = extract_features(file_path)
        if feat is not None:
            results.append(feat)
        
        # Augmented versions (noise + shift)
        for technique in ['noise', 'shift']:
            y_aug = augment_audio(y, sr, technique)
            with open('/tmp/aug_temp.wav', 'wb') as f:
                sf.write('/tmp/aug_temp.wav', y_aug, sr)
            feat_aug = extract_features('/tmp/aug_temp.wav')
            if feat_aug is not None:
                results.append(feat_aug)
    except Exception as e:
        print(f'Augmentation error: {e}')
    
    return results


print('Augmentation functions ready.')

## 5. Build Dataset

In [ ]:
from tqdm.notebook import tqdm  # pip install tqdm

X_list = []
y_list = []
skipped = 0

print(f'Processing {len(audio_files)} audio files...')

for file_path in tqdm(audio_files):
    # Parse RAVDESS filename
    parts = Path(file_path).stem.split('-')
    
    if len(parts) < 7:
        skipped += 1
        continue
    
    emotion_code = parts[2]
    emotion_label = EMOTION_MAP.get(emotion_code)
    
    if emotion_label is None:
        skipped += 1
        continue
    
    # Extract features (with augmentation)
    features_list = extract_with_augmentation(file_path)
    
    for feat in features_list:
        X_list.append(feat)
        y_list.append(emotion_label)


X = np.array(X_list)
y = np.array(y_list)

print(f'\nDataset shape: X={X.shape}, y={y.shape}')
print(f'Skipped files: {skipped}')
print(f'\nClass distribution:')
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f'  {label:12s}: {count}')

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 4))
labels_sorted = pd.Series(y).value_counts().index
counts_sorted = pd.Series(y).value_counts().values

colors = ['#fbbf24', '#60a5fa', '#f87171', '#a78bfa', '#94a3b8', '#86efac', '#fb923c', '#67e8f9']
bars = ax.bar(labels_sorted, counts_sorted, color=colors[:len(labels_sorted)], edgecolor='white', linewidth=0.5)

for bar, count in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Emotion Class Distribution in Dataset', fontsize=14, fontweight='bold', pad=16)
ax.set_ylabel('Sample Count')
ax.set_facecolor('#0e1221')
fig.patch.set_facecolor('#0e1221')
ax.tick_params(colors='white')
ax.title.set_color('white')
ax.yaxis.label.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2a3450')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Preprocessing: Encode & Scale

In [ ]:
# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot  = keras.utils.to_categorical(y_encoded)

print('Label encoding:')
for i, label in enumerate(le.classes_):
    print(f'  {i}: {label}')

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nScaled features: mean={X_scaled.mean():.4f}, std={X_scaled.std():.4f}')

# Train/validation/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_onehot, test_size=0.3, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f'\nSplit sizes: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')

# Reshape for LSTM: (samples, timesteps=1, features)
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_val   = X_val.reshape(X_val.shape[0],   1, X_val.shape[1])
X_test  = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print(f'LSTM input shape: {X_train.shape}')

## 7. LSTM Model Architecture

In [ ]:
NUM_CLASSES = len(le.classes_)
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])  # (1, 162)

def build_lstm_model(input_shape: tuple, num_classes: int) -> keras.Model:
    """
    CNN-LSTM hybrid architecture for speech emotion recognition.
    
    Architecture:
    Input → LSTM(256) → Dropout(0.3) → LSTM(128) → Dropout(0.3)
          → Dense(128, ReLU) → Dropout(0.3) → Dense(64, ReLU)
          → Dense(num_classes, Softmax)
    """
    inputs = keras.Input(shape=input_shape, name='audio_features')

    # First LSTM layer (return sequences for stacking)
    x = layers.LSTM(256, return_sequences=True, name='lstm_1')(
        inputs,
        # Recurrent dropout for regularization
    )
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Second LSTM layer
    x = layers.LSTM(128, return_sequences=False, name='lstm_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Dense classification head
    x = layers.Dense(128, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.2)(x)

    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax', name='emotion_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='VoiceIQ_LSTM')
    return model


model = build_lstm_model(INPUT_SHAPE, NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\nTotal trainable parameters: {model.count_params():,}')

## 8. Training

In [ ]:
# Training callbacks
callbacks = [
    # Stop early if no improvement
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    # Save best model
    ModelCheckpoint(
        filepath='../models/lstm_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

print('Starting training...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print(f'\nTraining complete!')
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')

## 9. Evaluation & Visualization

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0e1221')

for ax in axes:
    ax.set_facecolor('#141828')
    ax.tick_params(colors='#8b9bbe')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a3450')

# Accuracy
axes[0].plot(history.history['accuracy'],     color='#5b8ef0', linewidth=2, label='Train')
axes[0].plot(history.history['val_accuracy'], color='#0ef5c8', linewidth=2, label='Validation', linestyle='--')
axes[0].set_title('Model Accuracy', color='white', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy', color='#8b9bbe')
axes[0].set_xlabel('Epoch', color='#8b9bbe')
axes[0].legend(facecolor='#141828', labelcolor='white')

# Loss
axes[1].plot(history.history['loss'],     color='#f87171', linewidth=2, label='Train')
axes[1].plot(history.history['val_loss'], color='#fb923c', linewidth=2, label='Validation', linestyle='--')
axes[1].set_title('Model Loss', color='white', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Loss', color='#8b9bbe')
axes[1].set_xlabel('Epoch', color='#8b9bbe')
axes[1].legend(facecolor='#141828', labelcolor='white')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Test set evaluation
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_accuracy*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

# Predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Classification report
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0e1221')
ax.set_facecolor('#141828')

sns.heatmap(
    cm_normalized, annot=True, fmt='.2f',
    xticklabels=le.classes_, yticklabels=le.classes_,
    cmap='Blues', ax=ax,
    linewidths=0.5, linecolor='#2a3450'
)

ax.set_title('Normalized Confusion Matrix', color='white', fontsize=14, fontweight='bold', pad=16)
ax.set_xlabel('Predicted Label', color='#8b9bbe', labelpad=10)
ax.set_ylabel('True Label', color='#8b9bbe', labelpad=10)
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Model Artifacts

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Save the model (also saved by ModelCheckpoint callback above)
model.save('../models/lstm_model.h5')
print('✅ Model saved: ../models/lstm_model.h5')

# Save the scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Scaler saved: ../models/scaler.pkl')

# Save the label encoder
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('✅ Label encoder saved: ../models/label_encoder.pkl')

# Print final metrics
print(f'\n📊 Final Results:')
print(f'  Test Accuracy : {test_accuracy*100:.2f}%')
print(f'  Emotion Classes: {list(le.classes_)}')
print(f'  Feature Dim    : {X_train.shape[-1]}')
print(f'  Model Params   : {model.count_params():,}')